# HSJ Adversarial Robustness + ART BinaryInputDetector (Tabular Example)

This notebook demonstrates the full ART pipeline:

1. Train a **sklearn** base classifier on tabular data.
2. Wrap it in `SklearnClassifier`.
3. Generate adversarial examples with **HopSkipJump (HSJ)**.
4. Build a detection dataset: clean vs adversarial.
5. Train a **tf.keras** detector wrapped in `KerasClassifier`.
6. Use `BinaryInputDetector` to evaluate detection performance (TPR/FPR).


# New Section

In [8]:

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from art.estimators.classification import SklearnClassifier, KerasClassifier
from art.attacks.evasion import HopSkipJump
from art.defences.detector.evasion import BinaryInputDetector

from tensorflow.keras import models, layers



In [9]:
# import os
# print(os.getcwd())

df = pd.read_csv("CSVs/dataset.csv")

print(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# We assume:
#   - 'anomaly' is the label (0/1)
#   - 'train' = 1 → training, 0 → test
#   - 'channel' is a string; we drop it for now
#   - 'segment' is just an ID; we drop it
target_col = "anomaly"

feature_cols = [c for c in df.columns if c not in ["anomaly", "train", "channel", "segment"]]

train_df = df[df["train"] == 1].copy()
test_df  = df[df["train"] == 0].copy()

X_train = train_df[feature_cols].to_numpy().astype(np.float32)
y_train = train_df[target_col].to_numpy().astype(int)

X_test  = test_df[feature_cols].to_numpy().astype(np.float32)
y_test  = test_df[target_col].to_numpy().astype(int)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape :", X_test.shape, y_test.shape)


   segment  anomaly  train   channel  sampling  duration  len          mean  \
0        1        1      1  CADC0872         1       279  280  8.533143e-07   
1        2        1      1  CADC0872         1       476  477 -3.639396e-06   
2        3        1      1  CADC0872         1       594  595  1.170788e-05   
3        4        1      1  CADC0872         1       271  272  8.486808e-07   
4        5        0      0  CADC0872         1       255  257  1.058485e-05   

            var       std  ...  smooth10_n_peaks  smooth20_n_peaks  \
0  3.494283e-10  0.000019  ...                 3                 2   
1  6.476485e-10  0.000025  ...                 1                 1   
2  5.592877e-10  0.000024  ...                 2                 2   
3  5.466024e-10  0.000023  ...                 2                 2   
4  5.279023e-10  0.000023  ...                 1                 1   

   diff_peaks  diff2_peaks      diff_var     diff2_var  gaps_squared  \
0           4            6  1.27

In [10]:
base_model = LogisticRegression(max_iter=3000)
base_model.fit(X_train, y_train)

y_pred_test = base_model.predict(X_test)
clean_test_acc = accuracy_score(y_test, y_pred_test)
print(f"Base model clean test accuracy: {clean_test_acc:.4f}")


Base model clean test accuracy: 0.9130


C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
def make_art_base_classifier(sk_model, X_train):
    """
    Wraps a sklearn model so ART attacks can query it.
    """
    x_min = float(X_train.min())
    x_max = float(X_train.max())
    clip_values = (x_min, x_max)

    art_clf = SklearnClassifier(
        model=sk_model,
        clip_values=clip_values,
    )
    return art_clf, clip_values

def _predict_labels_art(art_clf, X):
    """
    Robust prediction: if ART returns probabilities (n, k), take argmax;
    if it returns labels (n,), use directly.
    """
    preds = np.asarray(art_clf.predict(X))
    if preds.ndim == 1:
        return preds.astype(int)
    return np.argmax(preds, axis=1)

art_clf, clip_values = make_art_base_classifier(base_model, X_train)
print("ART SklearnClassifier created with clip_values:", clip_values)


ART SklearnClassifier created with clip_values: (-1.8588135242462158, 20750.0)


In [12]:
def generate_hsj_adversarials(art_clf, X_test, y_test, max_samples=1000, hsj_kwargs=None):
    """
    Generate HSJ adversarials on a subset of test data and report robustness.
    """
    if hsj_kwargs is None:
        hsj_kwargs = dict(
            max_iter=20,
            max_eval=10000,
            init_eval=100,
            init_size=10,
            targeted=False,
            norm=2,
        )

    # Take subset
    n = min(max_samples, len(X_test))
    X = X_test[:n].astype(np.float32)
    y = y_test[:n].astype(int)

    # Encode labels to 0..K-1 for ART
    classes, y_int = np.unique(y, return_inverse=True)
    y_enc = y_int

    hsj = HopSkipJump(classifier=art_clf, **hsj_kwargs)

    print(f"Generating HSJ adversarials for {n} samples...")
    X_adv = hsj.generate(x=X, y=y_enc)

    # Robustness metrics
    y_pred_clean = _predict_labels_art(art_clf, X)
    y_pred_adv   = _predict_labels_art(art_clf, X_adv)

    clean_acc = accuracy_score(y_enc, y_pred_clean)
    adv_acc   = accuracy_score(y_enc, y_pred_adv)
    acc_drop  = clean_acc - adv_acc

    print(f"Clean accuracy (subset): {clean_acc:.4f}")
    print(f"Adv   accuracy (subset): {adv_acc:.4f}")
    print(f"Accuracy drop         : {acc_drop:.4f}")

    return X, y_enc, X_adv, classes, clean_acc, adv_acc, acc_drop

X_clean_sub, y_sub, X_adv_sub, classes, clean_acc, adv_acc, acc_drop = \
    generate_hsj_adversarials(art_clf, X_test, y_test, max_samples=2000)


Generating HSJ adversarials for 529 samples...


HopSkipJump: 100%|██████████| 529/529 [00:11<00:00, 47.58it/s]

Clean accuracy (subset): 0.9130
Adv   accuracy (subset): 0.0907
Accuracy drop         : 0.8223


In [13]:
def build_detector_dataset(X_clean, X_adv):
    """
    Construct dataset for the detector: clean vs adversarial.
    """
    X_clean_det = X_clean
    X_adv_det   = X_adv

    X_det = np.vstack([X_clean_det, X_adv_det])
    y_det = np.concatenate([
        np.zeros(len(X_clean_det), dtype=int),  # 0 = clean
        np.ones(len(X_adv_det), dtype=int),     # 1 = adversarial
    ])

    return X_det, y_det, X_clean_det, X_adv_det

X_det, y_det, X_clean_det, X_adv_det = build_detector_dataset(X_clean_sub, X_adv_sub)
print("Detector dataset built:", X_det.shape, y_det.shape)


Detector dataset built: (1058, 19) (1058,)


In [14]:
def make_keras_detector(input_dim, clip_values):
    """
    Small MLP detector: predicts clean (0) vs adversarial (1).
    """
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(2, activation="softmax"),  # 2 classes: clean/adv
    ])

    # Compile the Keras model BEFORE wrapping it with ART's KerasClassifier.
    # Use sparse_categorical_crossentropy since labels are integers (0/1).
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    det_art = KerasClassifier(
        model=model,
        clip_values=clip_values,
    )
    return det_art

input_dim = X_det.shape[1]
det_art = make_keras_detector(input_dim, clip_values)
print("Keras detector backend created with input_dim:", input_dim)


Keras detector backend created with input_dim: 19


In [15]:
def train_art_binary_input_detector(X_det, y_det, det_art, nb_epochs=10, batch_size=32):
    """
    Train ART's BinaryInputDetector on clean+adv dataset.
    """
    detector = BinaryInputDetector(detector=det_art)
    detector.fit(X_det, y_det, nb_epochs=nb_epochs, batch_size=batch_size)
    return detector

detector = train_art_binary_input_detector(X_det, y_det, det_art, nb_epochs=10, batch_size=32)
print("BinaryInputDetector trained.")


BinaryInputDetector trained.


In [16]:
def evaluate_detector(detector, X_clean_det, X_adv_det):
    """
    Compute TPR (on adversarial) and FPR (on clean).
    """
    _, clean_flags = detector.detect(X_clean_det)
    _, adv_flags   = detector.detect(X_adv_det)

    det_fpr = float(clean_flags.mean())  # fraction of clean flagged
    det_tpr = float(adv_flags.mean())    # fraction of adv correctly flagged

    print(f"Detector TPR (adv caught):    {det_tpr:.4f}")
    print(f"Detector FPR (clean flagged): {det_fpr:.4f}")
    return det_tpr, det_fpr

det_tpr, det_fpr = evaluate_detector(detector, X_clean_det, X_adv_det)

results = pd.DataFrame([{
    "Model": "LogisticRegression (anomaly)",
    "CleanAccSubset": clean_acc,
    "AdvAccSubset": adv_acc,
    "AccDropSubset": acc_drop,
    "DetectorTPR": det_tpr,
    "DetectorFPR": det_fpr,
    "NumAttacked": len(X_clean_sub),
}])

display(results)


Detector TPR (adv caught):    0.3913
Detector FPR (clean flagged): 0.3781


,Model,CleanAccSubset,AdvAccSubset,AccDropSubset,DetectorTPR,DetectorFPR,NumAttacked
0,LogisticRegression (anomaly),0.913043,0.090737,0.822306,0.391304,0.378072,529
